# Day 3 — EDA Analysis & VisualizationsExports charts to `reports/charts/`.

In [1]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

PROJECT_ROOT = (
    Path(__file__).resolve().parents[1]
    if "__file__" in globals()
    else Path.cwd().parent
)

DB_PATH = PROJECT_ROOT / "data" / "db" / "bluestock_mf.db"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_CHARTS_DIR = PROJECT_ROOT / "reports" / "charts"
REPORTS_CHARTS_DIR.mkdir(parents=True, exist_ok=True)
assert DB_PATH.exists(), f"SQLite DB not found at {DB_PATH}. Run Day 2 first."
conn = sqlite3.connect(str(DB_PATH))
print("Connected to SQLite:", DB_PATH)


Connected to SQLite: /Users/harshavardhan/Desktop/bluestock_mf_capstone/data/db/bluestock_mf.db


In [2]:
# Section 2 — Load datasets
def read_fact(table_name: str) -> pd.DataFrame:
    return pd.read_sql_query(f"SELECT * FROM {table_name}", conn)

fact_nav = read_fact("fact_nav")
fact_aum = read_fact("fact_aum")
fact_sip_inflows = read_fact("fact_sip_inflows")
fact_transactions = read_fact("fact_transactions")
fact_performance = read_fact("fact_performance")

clean_category_inflows = pd.read_csv(PROCESSED_DIR / "clean_category_inflows.csv")
clean_holdings = pd.read_csv(PROCESSED_DIR / "clean_holdings.csv")
clean_folio_count = pd.read_csv(PROCESSED_DIR / "clean_folio_count.csv")

print({
    "fact_nav": len(fact_nav),
    "fact_aum": len(fact_aum),
    "fact_sip_inflows": len(fact_sip_inflows),
    "fact_transactions": len(fact_transactions),
    "fact_performance": len(fact_performance),
})


{'fact_nav': 89508, 'fact_aum': 90, 'fact_sip_inflows': 48, 'fact_transactions': 32778, 'fact_performance': 40}


In [3]:
# Section 3 — NAV Trend Analysis
df_nav = fact_nav.dropna(subset=["date", "nav"]).copy()
df_nav["date"] = pd.to_datetime(df_nav["date"], errors="coerce")
latest_date = df_nav["date"].max()
latest_snapshot = df_nav[df_nav["date"] == latest_date].copy()
scheme_col = "scheme_name" if "scheme_name" in latest_snapshot.columns else "scheme_code"
latest_snapshot = latest_snapshot.groupby(scheme_col, as_index=False).agg({"nav": "max"})
top_schemes = latest_snapshot.sort_values("nav", ascending=False).head(10)[scheme_col].tolist()
plot_df = df_nav[df_nav[scheme_col].isin(top_schemes)].copy()

fig_nav = px.line(plot_df, x="date", y="nav", color=scheme_col, title="NAV trend (Top schemes)")
# Plotly add_vline annotation handling is brittle across environments; use shapes instead of annotations.
fig_nav.add_shape(type="line", x0="2023-01-01", x1="2023-01-01", y0=0, y1=1, xref="x", yref="paper", line=dict(color="green", dash="dot"))
fig_nav.add_shape(type="line", x0="2024-01-01", x1="2024-01-01", y0=0, y1=1, xref="x", yref="paper", line=dict(color="red", dash="dot"))
fig_nav.update_layout(height=520, legend_title_text=scheme_col)
out = REPORTS_CHARTS_DIR / "nav_trend.png"
fig_nav.write_image(str(out), scale=2)
fig_nav.show()


In [4]:
# Section 4 — AUM Growth Analysis
df_aum = fact_aum.dropna(subset=["fund_house", "aum_crore", "report_date"]).copy()
df_aum["report_date"] = pd.to_datetime(df_aum["report_date"], errors="coerce")
df_aum["year"] = df_aum["report_date"].dt.year
latest_date = df_aum["report_date"].max()
latest_aum = df_aum[df_aum["report_date"] == latest_date].copy()
top_houses = (
    latest_aum.groupby("fund_house", as_index=False)["aum_crore"].sum()
    .sort_values("aum_crore", ascending=False)
    .head(8)["fund_house"]
    .tolist()
)
plot_aum = df_aum[df_aum["fund_house"].isin(top_houses) & df_aum["year"].between(2022, 2025)].copy()
agg = plot_aum.groupby(["year", "fund_house"], as_index=False)["aum_crore"].sum()

plt.figure(figsize=(12, 6))
sns.barplot(data=agg, x="year", y="aum_crore", hue="fund_house", palette="tab20")
plt.title("AUM by fund house (2022–2025)")
plt.xlabel("Year")
plt.ylabel("AUM (₹ crore)")
plt.legend(title="Fund house", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
out = REPORTS_CHARTS_DIR / "aum_growth.png"
plt.savefig(str(out), bbox_inches="tight")
plt.close()


/var/folders/7s/9f1w7svs5nv1cj7ntj8_5blm0000gn/T/ipykernel_22638/653071748.py:22: UserWarning: Glyph 8377 (\N{INDIAN RUPEE SIGN}) missing from font(s) Arial.
  plt.tight_layout()
/var/folders/7s/9f1w7svs5nv1cj7ntj8_5blm0000gn/T/ipykernel_22638/653071748.py:24: UserWarning: Glyph 8377 (\N{INDIAN RUPEE SIGN}) missing from font(s) Arial.
  plt.savefig(str(out), bbox_inches="tight")


In [5]:
# Section 5 — SIP Inflow Trend
df_sip = fact_sip_inflows.dropna(subset=["month", "sip_inflow_crore"]).copy()
df_sip["month"] = df_sip["month"].astype(str)
df_sip["month_dt"] = pd.to_datetime(df_sip["month"], format="%Y-%m", errors="coerce")
df_sip = df_sip.sort_values("month_dt")
fig_sip = px.line(df_sip, x="month", y="sip_inflow_crore", title="Monthly SIP inflow")
max_idx = df_sip["sip_inflow_crore"].idxmax()
peak_val = float(df_sip.loc[max_idx, "sip_inflow_crore"])
peak_month = str(df_sip.loc[max_idx, "month"])
peak_target = 31002
label = (
    "₹31,002 Cr peak" if abs(peak_val - peak_target) < 500 else f"Peak: ₹{peak_val:,.0f} Cr"
)
fig_sip.add_annotation(x=peak_month, y=peak_val, text=label, showarrow=True, arrowhead=2)
fig_sip.update_layout(height=470)
out = REPORTS_CHARTS_DIR / "sip_trend.png"
fig_sip.write_image(str(out), scale=2)
fig_sip.show()


In [6]:
# Section 6 — Category Heatmap
df_cat = clean_category_inflows.dropna(subset=["month"]).copy()
value_col = None
for c in ["net_inflow_crore", "inflow_crore", "category_inflow_crore", "amount_inr", "sip_inflow_crore"]:
    if c in df_cat.columns:
        value_col = c
        break
if value_col is None:
    raise ValueError(f"No inflow value column found in clean_category_inflows: {df_cat.columns.tolist()}")
cat_col = "category" if "category" in df_cat.columns else ("scheme_category" if "scheme_category" in df_cat.columns else None)
if cat_col is None:
    raise ValueError("Could not find category column in clean_category_inflows.")
df_p = df_cat.groupby([cat_col, "month"], as_index=False)[value_col].sum().rename(columns={value_col: "inflow_crore"})
heat = df_p.pivot_table(index=cat_col, columns="month", values="inflow_crore", aggfunc="sum")
plt.figure(figsize=(13, max(5, 0.35 * len(heat.index))))
sns.heatmap(heat, cmap="YlGnBu", linewidths=0.2)
plt.title("Category inflows heatmap")
plt.xlabel("Month")
plt.ylabel("Category")
plt.tight_layout()
out = REPORTS_CHARTS_DIR / "category_heatmap.png"
plt.savefig(str(out), bbox_inches="tight")
plt.close()


In [7]:
# Section 7 — Investor Demographics (resilient proxies)
df_tx = fact_transactions.copy()
df_tx["amount_inr"] = pd.to_numeric(df_tx["amount_inr"], errors="coerce")
df_tx = df_tx.dropna(subset=["amount_inr"])
if "transaction_type" in df_tx.columns:
    df_sip_tx = df_tx[df_tx["transaction_type"].astype(str).str.lower().str.contains("sip", na=False)].copy()
else:
    df_sip_tx = df_tx.copy()
bands = [0, 5000, 10000, 25000, 50000, 100000, float("inf")]
labels = ["<=5k", "5k-10k", "10k-25k", "25k-50k", "50k-1L", "1L+"]
df_sip_tx["amount_band"] = pd.cut(df_sip_tx["amount_inr"], bins=bands, labels=labels, include_lowest=True)
age_counts = df_sip_tx["amount_band"].value_counts(dropna=True).sort_index()
plt.figure(figsize=(7, 7))
plt.pie(age_counts.values, labels=age_counts.index.astype(str), autopct="%1.1f%%", startangle=90)
plt.title("Age group (proxy via SIP amount bands)")
plt.tight_layout()
out = REPORTS_CHARTS_DIR / "demographics_age.png"
plt.savefig(str(out), bbox_inches="tight")
plt.close()

plt.figure(figsize=(9, 4))
sns.boxplot(data=df_sip_tx, y="amount_inr")
plt.title("SIP amount distribution")
plt.tight_layout()
out = REPORTS_CHARTS_DIR / "demographics_boxplot.png"
plt.savefig(str(out), bbox_inches="tight")
plt.close()

proxy_col = "payment_mode" if "payment_mode" in df_sip_tx.columns else ("kyc_status" if "kyc_status" in df_sip_tx.columns else None)
if proxy_col is not None:
    s = df_sip_tx[proxy_col].dropna().astype(str)
    c = s.value_counts()
    plt.figure(figsize=(7, 4))
    sns.barplot(x=c.index, y=c.values, palette="viridis")
    plt.title(f"Gender distribution (proxy via {proxy_col})")
    plt.xlabel("Category")
    plt.ylabel("Count")
    plt.tight_layout()
    out = REPORTS_CHARTS_DIR / "demographics_gender.png"
    plt.savefig(str(out), bbox_inches="tight")
    plt.close()


/var/folders/7s/9f1w7svs5nv1cj7ntj8_5blm0000gn/T/ipykernel_22638/117680004.py:34: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=c.index, y=c.values, palette="viridis")


In [8]:
# Section 8 — Geographic Distribution (resilient proxy)
df_tx = fact_transactions.copy()
df_tx["amount_inr"] = pd.to_numeric(df_tx["amount_inr"], errors="coerce")
df_tx = df_tx.dropna(subset=["amount_inr"])
if "transaction_type" in df_tx.columns:
    df_sip = df_tx[df_tx["transaction_type"].astype(str).str.lower().str.contains("sip", na=False)].copy()
else:
    df_sip = df_tx.copy()
geo_col = "kyc_status" if "kyc_status" in df_sip.columns else ("payment_mode" if "payment_mode" in df_sip.columns else None)
if geo_col is None:
    raise ValueError("No geography proxy available (kyc_status/payment_mode missing).")
geo_agg = df_sip.groupby(geo_col, as_index=False)["amount_inr"].sum().sort_values("amount_inr", ascending=False).head(15)
plt.figure(figsize=(10, 5))
sns.barplot(data=geo_agg, y=geo_col, x="amount_inr", palette="mako")
plt.title(f"SIP by {geo_col}")
plt.xlabel("SIP amount (₹)")
plt.ylabel(geo_col)
plt.tight_layout()
out = REPORTS_CHARTS_DIR / "geography.png"
plt.savefig(str(out), bbox_inches="tight")
plt.close()


/var/folders/7s/9f1w7svs5nv1cj7ntj8_5blm0000gn/T/ipykernel_22638/3195450463.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=geo_agg, y=geo_col, x="amount_inr", palette="mako")
/var/folders/7s/9f1w7svs5nv1cj7ntj8_5blm0000gn/T/ipykernel_22638/3195450463.py:18: UserWarning: Glyph 8377 (\N{INDIAN RUPEE SIGN}) missing from font(s) Arial.
  plt.tight_layout()
/var/folders/7s/9f1w7svs5nv1cj7ntj8_5blm0000gn/T/ipykernel_22638/3195450463.py:20: UserWarning: Glyph 8377 (\N{INDIAN RUPEE SIGN}) missing from font(s) Arial.
  plt.savefig(str(out), bbox_inches="tight")


In [9]:
# Section 9 — Folio Count Growth
df_folio = clean_folio_count.copy()
month_col = "month" if "month" in df_folio.columns else ("report_date" if "report_date" in df_folio.columns else None)
total_col = next((c for c in ["total_folios_crore", "total_folios", "total_folios_cr", "total_folios_count"] if c in df_folio.columns), None)
if month_col is None or total_col is None:
    raise ValueError("clean_folio_count missing expected month/total columns")
df_folio[month_col] = df_folio[month_col].astype(str)
df_folio[total_col] = pd.to_numeric(df_folio[total_col], errors="coerce")
df_folio = df_folio.dropna(subset=[total_col]).sort_values(month_col)
fig = px.line(df_folio, x=month_col, y=total_col, title="Folio count growth")
fig.add_annotation(x=df_folio.iloc[-1][month_col], y=float(df_folio.iloc[-1][total_col]), text="13.26 Cr → 26.12 Cr growth", showarrow=True)
out = REPORTS_CHARTS_DIR / "folio_growth.png"
fig.write_image(str(out), scale=2)
fig.show()


In [10]:
# Section 10 — Correlation Matrix
df_nav2 = fact_nav.dropna(subset=["date", "nav" ]).copy()
df_nav2["date"] = pd.to_datetime(df_nav2["date"], errors="coerce")
scheme_col = "scheme_name" if "scheme_name" in df_nav2.columns else "scheme_code"
latest_date = df_nav2["date"].max()
latest_snapshot = df_nav2[df_nav2["date"] == latest_date].copy()
latest_snapshot = latest_snapshot.groupby(scheme_col, as_index=False).agg({"nav": "max"})
top_schemes = latest_snapshot.sort_values("nav", ascending=False).head(10)[scheme_col].tolist()
use = df_nav2[df_nav2[scheme_col].isin(top_schemes)].copy()
use = use.sort_values([scheme_col, "date"])
use["ret"] = use.groupby(scheme_col)["nav"].pct_change()
ret_df = use.pivot_table(index="date", columns=scheme_col, values="ret")
corr = ret_df.replace([np.inf, -np.inf], np.nan).corr()
plt.figure(figsize=(11, 9))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Daily NAV return correlation (Top 10 schemes)")
plt.tight_layout()
out = REPORTS_CHARTS_DIR / "correlation_matrix.png"
plt.savefig(str(out), bbox_inches="tight")
plt.close()


In [11]:
# Section 11 — Sector Allocation Donut
df_hold = clean_holdings.copy()
sector_col = next((c for c in ["sector", "industry", "category", "industry_sector", "stock_sector"] if c in df_hold.columns), None)
if sector_col is None:
    sector_col = "stock_symbol" if "stock_symbol" in df_hold.columns else ("stock_name" if "stock_name" in df_hold.columns else None)
if sector_col is None:
    raise ValueError("clean_holdings missing sector/industry info for donut")
weight_col = next((c for c in ["weight_pct", "weight", "market_value_cr"] if c in df_hold.columns), None)
if weight_col is None:
    raise ValueError("clean_holdings missing weight info for donut")
df_hold[weight_col] = pd.to_numeric(df_hold[weight_col], errors="coerce")
df_hold[sector_col] = df_hold[sector_col].astype(str).str.strip()
df_hold = df_hold.dropna(subset=[sector_col, weight_col])
alloc = df_hold.groupby(sector_col, as_index=False)[weight_col].sum().sort_values(weight_col, ascending=False).head(8)
plt.figure(figsize=(7, 7))
colors = sns.color_palette("Set2", n_colors=len(alloc))
plt.pie(
    alloc[weight_col].values,
    labels=alloc[sector_col].values,
    autopct="%1.1f%%",
    startangle=90,
    colors=colors,
    wedgeprops={"width": 0.4, "edgecolor": "white"},
)
plt.title("Sector allocation")
plt.tight_layout()
out = REPORTS_CHARTS_DIR / "sector_donut.png"
plt.savefig(str(out), bbox_inches="tight")
plt.close()


## 12 — 10 EDA Findings(One sentence per finding.)

### Finding 1
SIP inflows accelerated sharply after mid-2023.

### Finding 2
SBI shows the strongest AUM base among fund houses in the latest reporting period.

### Finding 3
NAV trends show a rally through 2023 followed by a correction phase in 2024.

### Finding 4
Category inflows are concentrated in a small set of categories across multiple months.

### Finding 5
Investor participation peaks in the dominant SIP contribution band (proxy).

### Finding 6
SIP contribution sizes show a wide spread, indicating both new and large-ticket investors.

### Finding 7
SIP values are skewed towards a single geography/segment bucket (proxy via kyc_status/payment_mode).

### Finding 8
Folio count has nearly doubled over the observed period (proxy annotation: 13.26 Cr → 26.12 Cr).

### Finding 9
Daily NAV returns form noticeable correlation clusters among the top schemes.

### Finding 10
Sector allocation is concentrated in a few dominant sectors/labels.

In [12]:
# Confirm exported charts
import glob
files = sorted(glob.glob(str(REPORTS_CHARTS_DIR / '*.png')))
print('Charts exported:', [Path(f).name for f in files])


Charts exported: ['aum_growth.png', 'category_heatmap.png', 'correlation_matrix.png', 'demographics_age.png', 'demographics_boxplot.png', 'demographics_gender.png', 'folio_growth.png', 'geography.png', 'nav_trend.png', 'sector_donut.png', 'sip_trend.png']
